# 03 — Exploratory Data Analysis

**Input:** `data/processed/registrations.parquet`
**Outputs:** 8 PNGs in `docs/eda/` + findings in markdown cells below each plot

**Eight questions answered:**
1. How does global registration volume trend over time?
2. Which countries dominate total registrations?
3. How has the drivetrain mix shifted?
4. What does EV adoption look like per region?
5. Is there seasonality in registrations?
6. Which countries are growing fastest YoY?
7. Where is data coverage incomplete?
8. What does Germany BEV look like — with the Dec 2023 anomaly marked?

**Rules:**
- All plots read from `registrations.parquet` only. Never touch `data/raw/`.
- Sum of the 8 individual drivetrains = total registrations per country-month.
  (This is the working assumption throughout EDA. No external total column is used.)
- Save every plot as PNG at `scale=2` for readability.
- Write a one-sentence finding directly after each plot.

## Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

PROCESSED = Path("../data/processed/registrations.parquet")
EDA_DIR = Path("../docs/eda")
EDA_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROCESSED)
df["date"] = pd.to_datetime(df["date"])

EV_DRIVETRAINS = ["Battery electric", "Plug-in hybrid"]

print(f"Rows:       {len(df):,}")
print(f"Countries:  {df['iso3'].nunique()}")
print(f"Series:     {df.groupby(['iso3','drivetrain']).ngroups}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Drivetrains: {sorted(df['drivetrain'].unique())}")

Rows:       33,064
Countries:  57
Series:     304
Date range: 2005-01-01 to 2026-10-01
Drivetrains: ['Battery electric', 'Combustion (unspecified)', 'Diesel', 'Ethanol/petrol blend', 'Hybrid', 'Other', 'Petrol', 'Plug-in hybrid']


## Plot 1: Global registration volume over time

In [2]:
global_monthly = (
    df.groupby("date")["registrations"].sum().reset_index()
)

fig = px.line(
    global_monthly, x="date", y="registrations",
    title="Global Monthly Passenger-Car Registrations (all countries, all drivetrains)",
    labels={"date": "Month", "registrations": "Registrations"},
)
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.write_image(EDA_DIR / "01_global_trend.png", scale=2)
fig.show()

# Numeric summaries
peak_row = global_monthly.loc[global_monthly["registrations"].idxmax()]
print(f"Peak month: {peak_row['date'].date()}  ({peak_row['registrations']:,.0f} registrations)")
print(f"Latest month: {global_monthly['date'].max().date()}  "
      f"({global_monthly['registrations'].iloc[-1]:,.0f} registrations)")

Peak month: 2024-12-01  (6,944,585 registrations)
Latest month: 2026-10-01  (5,149,876 registrations)


**Finding 1:** "Global registrations grew from ~2.0M/month in 2005 to ~5.1M/month in 2026-10, peaking at 6.94M in December 2024. The series shows a sharp COVID-19 trough in April–May 2020 (visible as a distinct V-shape), followed by a strong recovery from 2021 and sustained growth through 2026. Seasonality is analyzed separately in Plot 5."

## Plot 2: Top 15 countries by cumulative registrations

In [3]:
top15 = (
    df.groupby(["iso3", "area"])["registrations"].sum()
    .reset_index()
    .nlargest(15, "registrations")
    .sort_values("registrations")
)

fig = px.bar(
    top15, x="registrations", y="area", orientation="h",
    title="Top 15 Countries by Cumulative Registrations (entire history)",
    labels={"registrations": "Total registrations", "area": "Country"},
)
fig.update_layout(template="plotly_white", height=500)
fig.write_image(EDA_DIR / "02_top15_countries.png", scale=2)
fig.show()

# Coverage check
top15_share = top15["registrations"].sum() / df["registrations"].sum()
print(f"Top-15 share of all registrations: {top15_share:.1%}")
top15_abs = top15.sort_values("registrations", ascending=False).copy()
top15_abs["share"] = top15_abs["registrations"] / df["registrations"].sum() * 100
print(top15_abs[["area", "registrations", "share"]].head(6).to_string(index=False))

Top-15 share of all registrations: 91.2%
         area  registrations     share
        China    299190691.0 34.150550
United States    249910994.0 28.525613
       Brazil     37073012.0  4.231628
        Japan     34460570.0  3.933436
      Germany     29671286.0  3.386772
        India     25172584.0  2.873276


**Finding 2:** "The top 15 countries account for 91.2% of all registrations. China and the US are the two dominant markets by a wide margin, followed by Brazil, Japan, Germany, and India in close succession — the global market is highly concentrated but not dominated by any single country outside the top two."

## Plot 3: Drivetrain mix shift (share % over time)

In [4]:
mix = (
    df.groupby(["date", "drivetrain"])["registrations"].sum()
    .reset_index()
)
mix["share"] = mix.groupby("date")["registrations"].transform(
    lambda x: x / x.sum() if x.sum() > 0 else 0
)

fig = px.area(
    mix, x="date", y="share", color="drivetrain",
    title="Drivetrain Mix Over Time (share of monthly registrations)",
    labels={"date": "Month", "share": "Share of registrations", "drivetrain": "Drivetrain"},
)
fig.update_layout(template="plotly_white", yaxis_tickformat=".0%")
fig.write_image(EDA_DIR / "03_drivetrain_mix.png", scale=2)
fig.show()

# EV share at start and end
ev_by_month = (
    mix[mix["drivetrain"].isin(EV_DRIVETRAINS)]
    .groupby("date")["share"].sum()
)
print(f"EV share — first month ({ev_by_month.index.min().date()}): {ev_by_month.iloc[0]:.2%}")
print(f"EV share — latest month ({ev_by_month.index.max().date()}): {ev_by_month.iloc[-1]:.2%}")

EV share — first month (2005-01-01): 0.01%
EV share — latest month (2026-10-01): 34.82%


**Finding 3:** [ "BEV+PHEV share rose from 0.01% (first month) to 34.82% (latest), with the steepest acceleration after 2020; petrol and diesel combined have fallen from ~Z% to ~W%."]

## Plot 4: EV adoption by region (small multiples)

In [5]:

# Calculate total registrations by region and month
region_total = (
    df.groupby(["ember_region", "date"])["registrations"]
    .sum()
    .reset_index(name="total")
)

# Calculate EV registrations (BEV + PHEV) by region and month
region_ev = (
    df[df["drivetrain"].isin(EV_DRIVETRAINS)]
    .groupby(["ember_region", "date"])["registrations"]
    .sum()
    .reset_index(name="ev")
)

# Combine EV registrations with total registrations
region_ev = region_total.merge(
    region_ev,
    on=["ember_region", "date"],
    how="left",
    validate="one_to_one"
)

# Calculate EV market share
region_ev["ev_share"] = np.where(
    region_ev["total"] > 0,
    region_ev["ev"] / region_ev["total"],
    np.nan
)

# Plot EV adoption by region
fig = px.line(
    region_ev,
    x="date",
    y="ev_share",
    facet_col="ember_region",
    facet_col_wrap=3,
    title="EV Share of Registrations by Ember Region",
    labels={
        "date": "Month",
        "ev_share": "EV share",
        "ember_region": "Region"
    },
)

fig.update_layout(
    template="plotly_white",
    height=600,
    yaxis_tickformat=".0%"
)

fig.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig.write_image(EDA_DIR / "04_ev_by_region.png", scale=2)
fig.show()

# Display latest EV share by region
latest_date = region_ev["date"].max()

print(f"Latest EV share by region ({latest_date.date()}):")

print(
    region_ev[region_ev["date"] == latest_date]
    .sort_values("ev_share", ascending=False)
    [["ember_region", "ev_share"]]
    .to_string(index=False)
)
print(region_ev.head())
print("Missing EV shares:", region_ev["ev_share"].isna().sum())
print("Duplicate region-date keys:",
      region_ev.duplicated(["ember_region", "date"]).sum())

Latest EV share by region (2026-10-01):
               ember_region  ev_share
                       Asia  0.463796
                Middle East  0.432537
                     Europe  0.428152
                    Oceania  0.322311
Latin America and Caribbean  0.123962
              North America  0.076406
  ember_region       date    total   ev  ev_share
0         Asia 2010-01-01  38566.0  0.0       0.0
1         Asia 2010-02-01  39305.0  0.0       0.0
2         Asia 2010-03-01  46302.0  0.0       0.0
3         Asia 2010-04-01  45507.0  0.0       0.0
4         Asia 2010-05-01  41280.0  0.0       0.0
Missing EV shares: 0
Duplicate region-date keys: 0


**Finding 4:** ["Asia leads EV adoption at 42.8% share as of 2026-10-01, followed by Asia-Pacific at 46.3%; North America at 7%. Adoption is uneven across regions and diverges sharply after 2020."]

## Plot 5: Seasonality of registrations

In [6]:
monthly = df.groupby("date")["registrations"].sum().reset_index()
monthly["year"] = monthly["date"].dt.year
monthly["month"] = monthly["date"].dt.month
monthly["month_name"] = monthly["date"].dt.strftime("%b")

# Index each month against its year's mean → isolates seasonal shape
monthly["year_mean"] = monthly.groupby("year")["registrations"].transform("mean")
monthly["index"] = monthly["registrations"] / monthly["year_mean"]

# Only use complete years (12 months present)
complete_years = monthly.groupby("year")["month"].nunique() == 12
valid_years = complete_years[complete_years].index
seasonality = monthly[monthly["year"].isin(valid_years)]

fig = px.box(
    seasonality, x="month_name", y="index", category_orders={"month_name": [
        "Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]},
    title=f"Seasonal Pattern (index = month / year-mean; complete years only: {list(valid_years)})",
    labels={"month_name": "Month", "index": "Registration index (1.0 = year average)"},
)
fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_white")
fig.write_image(EDA_DIR / "05_seasonality.png", scale=2)
fig.show()

# Month rankings by median index
med = seasonality.groupby("month_name")["index"].median().sort_values(ascending=False)
print("Median seasonal index by month (top to bottom):")
print(med.to_string())

Median seasonal index by month (top to bottom):
month_name
Dec    1.128442
Mar    1.078857
Jun    1.038763
Nov    1.017947
Oct    1.010221
May    1.002854
Sep    0.990893
Jan    0.975723
Apr    0.973888
Jul    0.965519
Aug    0.954535
Feb    0.842249


**Finding 5:** [ "March and December are the strongest registration months (median index 1.128), consistent with fiscal-year-end and plate-change effects in major markets; February and August are the weakest (~0.85)."]

## Plot 6: Top growing and declining countries (recent YoY)

In [7]:
# Use last 12 complete months vs prior 12 months, per country
monthly_country = df.groupby(["iso3", "area", "date"])["registrations"].sum().reset_index()
max_date = monthly_country["date"].max()
window_end = max_date
window_start = max_date - pd.DateOffset(months=11)
prev_end = window_start - pd.DateOffset(days=1)
prev_start = window_start - pd.DateOffset(months=12)

recent = monthly_country[(monthly_country["date"] >= window_start) & (monthly_country["date"] <= window_end)]
previous = monthly_country[(monthly_country["date"] >= prev_start) & (monthly_country["date"] <= prev_end)]

recent_sum = recent.groupby(["iso3", "area"])["registrations"].sum().rename("recent")
prev_sum = previous.groupby(["iso3", "area"])["registrations"].sum().rename("previous")

yoy = pd.concat([recent_sum, prev_sum], axis=1).dropna()
yoy = yoy[(yoy["recent"] + yoy["previous"]) >= 1000]  # ignore tiny markets for meaningfulness
yoy["yoy_pct"] = (yoy["recent"] - yoy["previous"]) / yoy["previous"].replace(0, np.nan) * 100
yoy = yoy.replace([np.inf, -np.inf], np.nan).dropna().reset_index()

top5 = yoy.nlargest(5, "yoy_pct")
bottom5 = yoy.nsmallest(5, "yoy_pct")
combined = pd.concat([bottom5, top5]).sort_values("yoy_pct")
combined["color"] = np.where(combined["yoy_pct"] >= 0, "growing", "declining")

fig = px.bar(
    combined, x="yoy_pct", y="area", orientation="h", color="color",
    color_discrete_map={"growing": "#2ca02c", "declining": "#d62728"},
    title=f"Top 5 Growing and Top 5 Declining Markets (YoY %, last 12 months vs prior 12 months, ending {max_date.date()})",
    labels={"yoy_pct": "YoY change (%)", "area": "Country"},
)
fig.update_layout(template="plotly_white", height=500, showlegend=False)
fig.write_image(EDA_DIR / "06_yoy_growth.png", scale=2)
fig.show()

**Finding 6:** [ "Markets Peru, Ecuador, Columbia grew fastest in the most recent 12-month window (up 34%–42%), while markets Turkiye and Taiwan declined 21%–29% — some of which reflect policy pull-forwards from the prior period."]

## Plot 7: Data coverage heatmap (country × year)

In [8]:
coverage = df[
    (~df["was_missing"]) &
    (df["registrations"].notna())
].copy()
coverage["year"] = coverage["date"].dt.year
cov_matrix = (
    coverage.groupby(["area", "year"])["date"].nunique()
    .reset_index()
    .pivot(index="area", columns="year", values="date")
    .fillna(0)
)

# Sort countries by first year of coverage, then alphabetically
cov_matrix = cov_matrix.loc[cov_matrix.apply(lambda r: r[r > 0].index.min() if (r > 0).any() else 9999, axis=1).sort_values().index]

fig = px.imshow(
    cov_matrix, aspect="auto",
    color_continuous_scale="Blues",
    title="Data Coverage — Months of Observation per Country per Year (0 = no data)",
    labels={"x": "Year", "y": "Country", "color": "Months observed"},
)
fig.update_layout(template="plotly_white", height=1000)
fig.write_image(EDA_DIR / "07_coverage_heatmap.png", scale=2)
fig.show()

# Countries with incomplete 2026
y2026 = cov_matrix.get(2026, pd.Series(dtype=float))
incomplete_2026 = y2026[y2026 < y2026.max()].sort_values()
print(f"2026 max months observed: {y2026.max()}")
print(f"Countries with < {y2026.max()} months in 2026: {len(incomplete_2026)}")
print(incomplete_2026.head(20).to_string())

2026 max months observed: 10.0
Countries with < 10.0 months in 2026: 27
area
Taiwan (China)              6.0
Brazil                      7.0
Hong Kong (SAR of China)    8.0
Ukraine                     8.0
Bulgaria                    8.0
Türkiye                     8.0
Ireland                     9.0
Luxembourg                  9.0
Mexico                      9.0
Belgium                     9.0
Thailand                    9.0
South Korea                 9.0
Austria                     9.0
Latvia                      9.0
Cyprus                      9.0
Iceland                     9.0
Lithuania                   9.0
Greece                      9.0
Croatia                     9.0
Estonia                     9.0


**Finding 7:** ["Coverage is heterogeneous: 2 countries have complete monthly data from 2005, Y countries start in 2023, and 27 countries have fewer than 10 months in 2026 — the latest month per country must therefore be tracked per-series, not assumed globally."]

## Plot 8: Germany BEV anomaly case study

In [9]:
de_bev = df[(df["iso3"] == "DEU") & (df["drivetrain"] == "Battery electric")]
de_bev = de_bev.sort_values("date")

fig = px.line(
    de_bev, x="date", y="registrations",
    title="Germany — Battery Electric Vehicle Registrations (Umweltbonus termination, Dec 2023)",
    labels={"date": "Month", "registrations": "Registrations"},
)
fig.add_vline(x=pd.Timestamp("2023-12-01"), line_dash="dash", line_color="red",
              annotation_text="Umweltbonus ends 17 Dec 2023",
              annotation_position="top right")
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.write_image(EDA_DIR / "08_germany_bev_anomaly.png", scale=2)
fig.show()

dec_2023 = de_bev.loc[de_bev["date"] == pd.Timestamp("2023-12-01"), "registrations"].iloc[0]
jan_2024 = de_bev.loc[de_bev["date"] == pd.Timestamp("2024-01-01"), "registrations"].iloc[0]
print(f"Dec 2023: {dec_2023:,.0f}")
print(f"Jan 2024: {jan_2024:,.0f}")
print(f"Drop:     {(jan_2024 - dec_2023) / dec_2023:.1%}")

Dec 2023: 54,654
Jan 2024: 22,474
Drop:     -58.9%


**Finding 8:** ["Germany BEV registrations collapsed X% from Dec 2023 (A) to Jan 2024 (B) following the Umweltbonus termination on 17 Dec 2023 — a genuine structural break that any autoregressive model must account for, either by excluding the pull-forward month from training or by adding an event dummy."]
The current notebook reports 54,654 and 22,474, not the older figures of 104,325 and 18,136.

# Summary (markdown)

## Summary of EDA Findings

| # | Question | Headline |
|---|---|---|
| 1 | Global volume trend | [fill] |
| 2 | Top countries | [fill] |
| 3 | Drivetrain mix | [fill] |
| 4 | EV by region | [fill] |
| 5 | Seasonality | [fill] |
| 6 | Fastest growing/declining | [fill] |
| 7 | Coverage gaps | [fill] |
| 8 | Germany anomaly | [fill] |

**Artifacts:** 8 PNGs in `docs/eda/` — committed. Notebook outputs preserved.

**Downstream consumers:**
- Plot 1, 2 → Global Overview page
- Plot 3, 4 → Global Market Intelligence page
- Plot 5 → Feature engineering (month/quarter features)
- Plot 6 → "Emerging opportunities" insight card
- Plot 7 → Data Quality / Data Explorer page
- Plot 8 → AI Insights page (structural-break explanation)